In [41]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from datasets import load_dataset
import pandas as pd

# Configuration
base_model_path = "./pretrain/Qwen1.5-1.8B-Chat"  # Path to your local base model
adapter_path = "./pretrain/final_poetry_adapter" # Path to your fine-tuned adapter
test_data_path = "./data/tang_poems/test-00000-of-00001-a794cd4c018c9326.parquet"
num_test_samples = 5 # Number of test samples to evaluate
max_new_tokens_generation = 50 # Max tokens to generate for the poem completion

In [42]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# QLoRA config for loading base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Base model loaded.")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Base model loaded.


In [43]:
# Load the LoRA adapter and merge it into the base model
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_path)
fine_tuned_model = fine_tuned_model.merge_and_unload() # Optional: merge to make inference faster, but consumes more memory if not careful
print("Fine-tuned model (base + adapter) loaded and merged.")

Fine-tuned model (base + adapter) loaded and merged.


In [44]:
import re

# Load the test dataset
test_dataset_full = load_dataset("parquet", data_files={"test": test_data_path}, split="test")

# Select a few samples
test_samples = test_dataset_full.shuffle(seed=42).select(range(num_test_samples))
def create_inference_prompt(sample_content):
    """
    Creates a prompt for inference based on the poem's first part.
    Uses the "paragraphs" key which your data loading logic implies.
    """
    poem_text = sample_content

    if isinstance(poem_text, list):
        poem_text = "".join(poem_text)
    elif not isinstance(poem_text, str):
        print(f"Warning: create_inference_prompt received non-string, non-list content: {type(poem_text)}")
        return None, None, None

    poem_text = poem_text.strip()
    if not poem_text:
        return None, None, None

    parts = re.split(r'([，。？！])', poem_text)
    if len(parts) < 3:
        return None, None, None
    
    prompt_starter_text = parts[0] + parts[1]
    actual_completion = "".join(parts[2:]).strip()
    
    # 构建 messages 列表
    # 确保这里的键名是 "content"
    messages = [
        {"role": "user", "content": f"你是一位唐诗专家，请创作以此句开头的这首唐诗：{prompt_starter_text}注意字数和格律"}
    ]
    
    # --- 调试代码开始 ---
    print("------------- DEBUG: messages content before apply_chat_template -------------")
    print(f"Type of messages: {type(messages)}")
    print(f"Content of messages: {messages}")
    if messages and isinstance(messages, list) and isinstance(messages[0], dict):
        print(f"Keys in first message dict: {messages[0].keys()}")
    print("--------------------------------------------------------------------------")
    # --- 调试代码结束 ---

    try:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return formatted_prompt, prompt_starter_text, actual_completion
    except Exception as e:
        print(f"Error during apply_chat_template: {e}") # 打印具体的模板错误
        # 可以在这里重新抛出异常或返回 None，以便了解错误详情
        raise # 或者 return None, None, None
prompts_data = []
print("--- Starting to process test_samples ---") # 新增
for i, sample in enumerate(test_samples): # 新增 enumerate 获取索引
    print(f"\nProcessing sample {i}...") # 新增
    content_list_or_str = sample['paragraphs']

    if isinstance(content_list_or_str, list):
        content_str = "".join(content_list_or_str)
    elif isinstance(content_list_or_str, str):
        content_str = content_list_or_str
    else:
        print(f"Skipping sample {i} due to unexpected content type: {type(content_list_or_str)}. Original sample data: {sample}") # 修改
        continue

    # 尝试调用 create_inference_prompt 并捕获可能的错误
    try:
        # 确保 create_inference_prompt 内部的 print 依然存在
        formatted_prompt, starter, actual_completion = create_inference_prompt(content_str)
        
        if formatted_prompt:
            prompts_data.append({
                "formatted_prompt": formatted_prompt,
                "starter_text": starter,
                "actual_completion": actual_completion,
                "full_original_poem": content_str
            })
            print(f"Sample {i} processed successfully.") # 新增
        else:
            print(f"Sample {i} resulted in None formatted_prompt. Original content_str: '{content_str}'") # 新增
            
    except Exception as e:
        print(f"!!!!!!!! ERROR occurred while processing sample {i} !!!!!!!!") # 新增
        print(f"Original content_str that caused error: '{content_str}'") # 新增
        print(f"Type of error: {type(e)}") # 新增
        print(f"Error message: {e}") # 新增
        # 如果需要完整的 traceback，可以取消下面一行的注释
        # import traceback
        # traceback.print_exc()
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!") # 新增
        # 您可以选择在这里停止循环 (break) 或继续处理其他样本 (continue)
        # break # 如果想在第一个错误处停止

print(f"\n--- Finished processing test_samples ---") # 新增
print(f"Prepared {len(prompts_data)} prompts for testing in total.")
if prompts_data:
    print("\nExample prompt data (first successfully processed sample, if any):")
    print(prompts_data[0])
else:
    print("No prompts were successfully prepared for testing.")

--- Starting to process test_samples ---

Processing sample 0...
------------- DEBUG: messages content before apply_chat_template -------------
Type of messages: <class 'list'>
Content of messages: [{'role': 'user', 'content': '你是一位唐诗专家，请创作以此句开头的这首唐诗：廷评年少法家流，注意字数和格律'}]
Keys in first message dict: dict_keys(['role', 'content'])
--------------------------------------------------------------------------
Sample 0 processed successfully.

Processing sample 1...
------------- DEBUG: messages content before apply_chat_template -------------
Type of messages: <class 'list'>
Content of messages: [{'role': 'user', 'content': '你是一位唐诗专家，请创作以此句开头的这首唐诗：枯木藏龙，注意字数和格律'}]
Keys in first message dict: dict_keys(['role', 'content'])
--------------------------------------------------------------------------
Sample 1 processed successfully.

Processing sample 2...
------------- DEBUG: messages content before apply_chat_template -------------
Type of messages: <class 'list'>
Content of messages: [{'role': 'us

In [47]:
# Create pipelines for generation
pipe_base = pipeline("text-generation", model=base_model, tokenizer=tokenizer, device_map="auto")
pipe_fine_tuned = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer, device_map="auto")

generation_params = {
    "max_new_tokens": max_new_tokens_generation,
    "do_sample": True,
    "temperature": 0.7,
    # "top_k": 50,
    # "top_p": 0.95,
    "pad_token_id": tokenizer.eos_token_id # Important for consistent generation
}

results = []

for item in prompts_data:
    print(f"\n--- Testing with prompt starter: {item['starter_text']} ---")
    
    # Generate with base model
    base_output = pipe_base(item['formatted_prompt'], **generation_params)
    base_generated_text = base_output[0]['generated_text']
    # Extract assistant's response for base model
    try:
        base_completion = base_generated_text.split("<|im_start|>assistant\n")[1].split("<|im_end|>")[0].strip()
    except IndexError:
        base_completion = "[Error extracting base model completion]"
        print(f"Base model raw output: {base_generated_text}")


    # Generate with fine-tuned model
    fine_tuned_output = pipe_fine_tuned(item['formatted_prompt'], **generation_params)
    fine_tuned_generated_text = fine_tuned_output[0]['generated_text']
    # Extract assistant's response for fine-tuned model
    try:
        fine_tuned_completion = fine_tuned_generated_text.split("<|im_start|>assistant\n")[1].split("<|im_end|>")[0].strip()
    except IndexError:
        fine_tuned_completion = "[Error extracting fine-tuned model completion]"
        print(f"Fine-tuned model raw output: {fine_tuned_generated_text}")

    results.append({
        "Starter": item['starter_text'],
        "Actual Completion": item['actual_completion'],
        "Base Model Output": base_completion,
        "Fine-tuned Model Output": fine_tuned_completion,
        "Full Original Poem": item['full_original_poem']
    })

# Display results in a DataFrame for easier comparison
df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None) # Show full text in cells
print("\n\n--- Comparison Results ---")
display(df_results)


--- Testing with prompt starter: 廷评年少法家流， ---


RuntimeError: probability tensor contains either `inf`, `nan` or element < 0